# Chapter 6: Batch TD vs Monte Carlo on the Random Walk

In [ ]:
import numpy as np

true_values = np.array([0, 1/6, 2/6, 3/6, 4/6, 5/6, 0])

def generate_episode(rng):
    s = 3
    episode = []
    while s not in (0, 6):
        ns = s + rng.choice([-1, 1])
        reward = 1.0 if ns == 6 else 0.0
        episode.append((s, reward, ns))
        s = ns
    return episode

def batch_td(episodes, alpha=0.01, sweeps=100):
    v = np.full(7, 0.5)
    v[0] = v[6] = 0.0
    for _ in range(sweeps):
        delta = np.zeros(7)
        for ep in episodes:
            for s, r, ns in ep:
                delta[s] += alpha * (r + v[ns] - v[s])
        v += delta
        v[0] = v[6] = 0.0
    return v

def batch_mc(episodes, alpha=0.01, sweeps=100):
    v = np.full(7, 0.5)
    v[0] = v[6] = 0.0
    for _ in range(sweeps):
        delta = np.zeros(7)
        for ep in episodes:
            G = 0.0
            returns = []
            for s, r, ns in reversed(ep):
                G = r + G
                returns.append((s, G))
            for s, G in returns:
                delta[s] += alpha * (G - v[s])
        v += delta
        v[0] = v[6] = 0.0
    return v

rng = np.random.default_rng(1)
episodes = [generate_episode(rng) for _ in range(10)]
v_td = batch_td(episodes)
v_mc = batch_mc(episodes)
rmse_td = np.sqrt(np.mean((v_td[1:6] - true_values[1:6])**2))
rmse_mc = np.sqrt(np.mean((v_mc[1:6] - true_values[1:6])**2))
v_td, v_mc, rmse_td, rmse_mc